## kernel_01_load.py

In [ ]:
# ================================================
# KERNEL 1 — Imports & Load Data
# ================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('online_retail.csv', encoding='latin1')

print("Dataset loaded successfully")
print(f"Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns : {df.columns.tolist()}")


## kernel_02_first_look.py

In [ ]:
# ================================================
# KERNEL 2 — First Look at the Data
# ================================================

# First 5 rows
print("=== First 5 Rows ===")
print(df.head())

# Data types
print("\n=== Data Types ===")
print(df.dtypes)

# Basic info
print("\n=== Dataset Info ===")
print(df.info())


## kernel_03_missing.py

In [ ]:
# ================================================
# KERNEL 3 — Missing Values
# ================================================

missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count' : missing,
    'Missing %'     : missing_pct
})

print("=== Missing Values ===")
print(missing_df[missing_df['Missing Count'] > 0])

# CustomerID has 135,080 missing (24.9%) — big problem
# Description has 1,454 missing  (0.3%)  — small, fixable


## kernel_04_stats.py

In [ ]:
# ================================================
# KERNEL 4 — Basic Statistics
# ================================================

print("=== Numeric Column Statistics ===")
print(df[['Quantity', 'UnitPrice']].describe())

print("\n=== Quantity Range ===")
print(f"Min Quantity : {df['Quantity'].min()}")
print(f"Max Quantity : {df['Quantity'].max()}")

print("\n=== UnitPrice Range ===")
print(f"Min UnitPrice : {df['UnitPrice'].min()}")
print(f"Max UnitPrice : {df['UnitPrice'].max()}")

# Spotted problems:
# Quantity min = -80,995  → returns/cancellations
# UnitPrice min = -11,062 → invalid, negative price
# UnitPrice = 0           → free items or errors


## kernel_05_duplicates.py

In [ ]:
# ================================================
# KERNEL 5 — Duplicates & Special Values
# ================================================

# Duplicate rows
print(f"Duplicate rows      : {df.duplicated().sum():,}")

# Negative values
print(f"Negative Quantity   : {(df['Quantity'] < 0).sum():,}")
print(f"Zero UnitPrice      : {(df['UnitPrice'] == 0).sum():,}")
print(f"Negative UnitPrice  : {(df['UnitPrice'] < 0).sum():,}")

# Cancelled orders (Invoice starts with 'C')
cancelled = df['InvoiceNo'].astype(str).str.startswith('C').sum()
print(f"Cancelled orders    : {cancelled:,}")

# Non-product StockCodes (letters only like POST, DOT, BANK)
non_product = df['StockCode'].astype(str).str.match(r'^[A-Za-z]+$').sum()
print(f"Non-product codes   : {non_product:,}")

print("\nSample cancelled orders:")
print(df[df['InvoiceNo'].astype(str).str.startswith('C')].head(3))

print("\nSample non-product StockCodes:")
print(df[df['StockCode'].astype(str).str.match(r'^[A-Za-z]+$')]['StockCode'].unique()[:8])


## kernel_06_dates_countries.py

In [ ]:
# ================================================
# KERNEL 6 — Date Range & Country Distribution
# ================================================

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print("=== Date Range ===")
print(f"From : {df['InvoiceDate'].min()}")
print(f"To   : {df['InvoiceDate'].max()}")

print("\n=== Unique Counts ===")
print(f"Unique Customers : {df['CustomerID'].nunique():,}")
print(f"Unique Products  : {df['StockCode'].nunique():,}")
print(f"Unique Invoices  : {df['InvoiceNo'].nunique():,}")
print(f"Unique Countries : {df['Country'].nunique():,}")

print("\n=== Top 10 Countries by Orders ===")
print(df['Country'].value_counts().head(10))


## kernel_07_monthly_revenue.py

In [ ]:
# ================================================
# KERNEL 7 — EDA Chart: Monthly Revenue
# ================================================

# Working copy — positive values only, no missing CustomerID
df_eda = df.copy()
df_eda = df_eda[(df_eda['Quantity'] > 0) & (df_eda['UnitPrice'] > 0)]
df_eda = df_eda.dropna(subset=['CustomerID'])
df_eda['TotalPrice'] = df_eda['Quantity'] * df_eda['UnitPrice']
df_eda['MonthYear']  = df_eda['InvoiceDate'].dt.to_period('M').astype(str)

monthly = df_eda.groupby('MonthYear').agg(
    Revenue = ('TotalPrice', 'sum'),
    Orders  = ('InvoiceNo',  'count')
).reset_index()

ORANGE  = '#E84E1B'
ORANGE2 = '#F7941D'
DARK    = '#1A1A1A'
GRAY    = '#666666'

fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('white')

clrs = [ORANGE if v == monthly['Revenue'].max()
        else '#cccccc' if v == monthly['Revenue'].min()
        else ORANGE2 for v in monthly['Revenue']]

bars = ax.bar(monthly['MonthYear'], monthly['Revenue'] / 1000,
              color=clrs, alpha=0.88, width=0.65, edgecolor='white')

# trend line
z = np.polyfit(range(len(monthly)), monthly['Revenue'] / 1000, 1)
ax.plot(monthly['MonthYear'], np.poly1d(z)(range(len(monthly))),
        '--', color=DARK, linewidth=1.5, alpha=0.6, label='Trend')

for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 3,
            f'£{bar.get_height():.0f}K',
            ha='center', va='bottom', fontsize=8, color=DARK)

ax.set_title('Monthly Revenue (£ Thousands) — Peak bar in orange', fontsize=13,
             fontweight='bold', color=DARK, pad=10)
ax.set_xlabel('Month–Year', color=GRAY)
ax.set_ylabel('Revenue (£K)', color=GRAY)
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('chart_monthly_revenue.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: chart_monthly_revenue.png")
print(f"\nTotal Revenue : £{df_eda['TotalPrice'].sum()/1e6:.2f}M")
print(f"Peak Month    : {monthly.loc[monthly['Revenue'].idxmax(), 'MonthYear']}")
print(f"Peak Revenue  : £{monthly['Revenue'].max()/1000:.0f}K")


## kernel_08_country_dow.py

In [ ]:
# ================================================
# KERNEL 8 — EDA Chart: Country & Day of Week
# ================================================

ORANGE  = '#E84E1B'
ORANGE2 = '#F7941D'
YELLOW  = '#FBBA13'
BLUE    = '#2E86AB'
PURPLE  = '#7B4D8E'
GREEN   = '#44BBA4'
RED     = '#C73E1D'
DARK    = '#1A1A1A'
GRAY    = '#666666'
PALETTE = [ORANGE, ORANGE2, YELLOW, BLUE, PURPLE, GREEN, RED, '#3B1F2B']

df_eda['DayOfWeek'] = df_eda['InvoiceDate'].dt.day_name()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('white')

# ── Left: Top 10 countries by revenue ──
top_ctry  = df_eda.groupby('Country')['TotalPrice'].sum().nlargest(10)
co_colors = [ORANGE, ORANGE2, YELLOW] + ['#cccccc'] * 7
axes[0].barh(top_ctry.index[::-1], top_ctry.values[::-1] / 1000,
             color=co_colors[::-1])
axes[0].set_title('Top 10 Countries by Revenue (£K)',
                  fontsize=12, fontweight='bold', color=DARK)
axes[0].set_xlabel('Revenue (£K)', color=GRAY)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# ── Right: Orders by day of week ──
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = df_eda['DayOfWeek'].value_counts().reindex(day_order).fillna(0)
bars_dw = axes[1].bar(dow.index, dow.values, color=PALETTE[:7], alpha=0.9, width=0.65)
axes[1].set_title('Orders by Day of Week',
                  fontsize=12, fontweight='bold', color=DARK)
axes[1].set_xlabel('Day', color=GRAY)
axes[1].set_ylabel('Number of Orders', color=GRAY)
axes[1].tick_params(axis='x', rotation=40)
for bar in bars_dw:
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 200,
                 f'{bar.get_height():,.0f}',
                 ha='center', va='bottom', fontsize=8, color=DARK)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('chart_country_dayofweek.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: chart_country_dayofweek.png")
print(f"\nTop Country : UK — {df_eda['Country'].value_counts().iloc[0]:,} orders")
print(f"Busiest Day : {dow.idxmax()} — {int(dow.max()):,} orders")
print(f"Quietest Day: {dow.idxmin()} — {int(dow.min()):,} orders")


## kernel_09_distributions.py

In [ ]:
# ================================================
# KERNEL 9 — EDA Chart: Quantity & Price Distributions
# ================================================

ORANGE  = '#E84E1B'
ORANGE2 = '#F7941D'
YELLOW  = '#FBBA13'
DARK    = '#1A1A1A'
GRAY    = '#666666'

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('white')

# ── Quantity distribution (capped at 50) ──
axes[0].hist(df_eda[df_eda['Quantity'] <= 50]['Quantity'],
             bins=30, color=ORANGE, alpha=0.85, edgecolor='white')
axes[0].axvline(df_eda['Quantity'].median(), color=DARK, linestyle='--',
                linewidth=1.5, label=f'Median = {df_eda["Quantity"].median():.0f}')
axes[0].set_title('Quantity per Order (capped at 50)',
                  fontsize=12, fontweight='bold', color=DARK)
axes[0].set_xlabel('Quantity', color=GRAY)
axes[0].set_ylabel('Frequency', color=GRAY)
axes[0].legend(fontsize=9)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# ── Unit price distribution (capped at £20) ──
axes[1].hist(df_eda[df_eda['UnitPrice'] <= 20]['UnitPrice'],
             bins=40, color=ORANGE2, alpha=0.85, edgecolor='white')
axes[1].axvline(df_eda['UnitPrice'].median(), color=DARK, linestyle='--',
                linewidth=1.5, label=f'Median = £{df_eda["UnitPrice"].median():.2f}')
axes[1].set_title('Unit Price Distribution (capped at £20)',
                  fontsize=12, fontweight='bold', color=DARK)
axes[1].set_xlabel('Unit Price (£)', color=GRAY)
axes[1].set_ylabel('Frequency', color=GRAY)
axes[1].legend(fontsize=9)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

# ── Total price distribution (capped at £100) ──
axes[2].hist(df_eda[df_eda['TotalPrice'] <= 100]['TotalPrice'],
             bins=50, color=YELLOW, alpha=0.85, edgecolor='white')
axes[2].axvline(df_eda['TotalPrice'].median(), color=DARK, linestyle='--',
                linewidth=1.5, label=f'Median = £{df_eda["TotalPrice"].median():.2f}')
axes[2].set_title('Total Price per Order (capped at £100)',
                  fontsize=12, fontweight='bold', color=DARK)
axes[2].set_xlabel('Total Price (£)', color=GRAY)
axes[2].set_ylabel('Frequency', color=GRAY)
axes[2].legend(fontsize=9)
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('chart_distributions.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: chart_distributions.png")
print(f"\nQuantity  — Mean: {df_eda['Quantity'].mean():.1f}, Median: {df_eda['Quantity'].median():.0f}")
print(f"UnitPrice — Mean: £{df_eda['UnitPrice'].mean():.2f}, Median: £{df_eda['UnitPrice'].median():.2f}")
print(f"TotalPrice— Mean: £{df_eda['TotalPrice'].mean():.2f}, Median: £{df_eda['TotalPrice'].median():.2f}")


## kernel_10_rfm.py

In [ ]:
# ================================================
# KERNEL 10 — EDA Chart: RFM Customer Analysis
# ================================================
# RFM = Recency, Frequency, Monetary
# Standard way to understand customer behaviour

ORANGE  = '#E84E1B'
ORANGE2 = '#F7941D'
YELLOW  = '#FBBA13'
DARK    = '#1A1A1A'
GRAY    = '#666666'
PALETTE = [ORANGE, ORANGE2, YELLOW, '#2E86AB', '#7B4D8E', '#44BBA4', '#C73E1D']

snapshot = df_eda['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df_eda.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot - pd.to_datetime(x).max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('TotalPrice',  'sum')
).reset_index()

rfm['Segment'] = pd.cut(rfm['Recency'],
                         bins=[0, 30, 90, 180, 400],
                         labels=['Active', 'At Risk', 'Lapsing', 'Lost'])

fig, axes = plt.subplots(2, 3, figsize=(20, 11))
fig.patch.set_facecolor('white')
fig.suptitle('Customer RFM Analysis', fontsize=16, fontweight='bold', color=DARK, y=1.01)

# Recency histogram
axes[0,0].hist(rfm['Recency'], bins=40, color=ORANGE, alpha=0.85, edgecolor='white')
axes[0,0].axvline(rfm['Recency'].median(), color=DARK, linestyle='--',
                  label=f'Median = {rfm["Recency"].median():.0f} days')
axes[0,0].set_title('Recency — Days Since Last Purchase', fontweight='bold', color=DARK)
axes[0,0].set_xlabel('Days', color=GRAY)
axes[0,0].set_ylabel('Customers', color=GRAY)
axes[0,0].legend(fontsize=9)

# Frequency histogram
axes[0,1].hist(rfm[rfm['Frequency'] <= 30]['Frequency'], bins=30,
               color=ORANGE2, alpha=0.85, edgecolor='white')
axes[0,1].axvline(rfm['Frequency'].median(), color=DARK, linestyle='--',
                  label=f'Median = {rfm["Frequency"].median():.0f}')
axes[0,1].set_title('Frequency — Orders per Customer (capped 30)',
                    fontweight='bold', color=DARK)
axes[0,1].set_xlabel('Number of Orders', color=GRAY)
axes[0,1].set_ylabel('Customers', color=GRAY)
axes[0,1].legend(fontsize=9)

# Monetary histogram
axes[0,2].hist(rfm[rfm['Monetary'] <= 5000]['Monetary'], bins=40,
               color=YELLOW, alpha=0.85, edgecolor='white')
axes[0,2].axvline(rfm['Monetary'].median(), color=DARK, linestyle='--',
                  label=f'Median = £{rfm["Monetary"].median():.0f}')
axes[0,2].set_title('Monetary — Total Spend per Customer (≤ £5K)',
                    fontweight='bold', color=DARK)
axes[0,2].set_xlabel('Total Spend (£)', color=GRAY)
axes[0,2].set_ylabel('Customers', color=GRAY)
axes[0,2].legend(fontsize=9)

# Segments donut
seg_c = rfm['Segment'].value_counts()
wedges, texts, autotexts = axes[1,0].pie(
    seg_c.values, labels=seg_c.index,
    colors=[ORANGE, ORANGE2, YELLOW, '#cccccc'],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    pctdistance=0.82)
for at in autotexts: at.set_fontsize(9)
centre = plt.Circle((0, 0), 0.55, fc='white')
axes[1,0].add_artist(centre)
axes[1,0].text(0, 0, f"{rfm['CustomerID'].nunique():,}\nCustomers",
               ha='center', va='center', fontsize=10, fontweight='bold', color=DARK)
axes[1,0].set_title('Customer Segments by Recency', fontweight='bold', color=DARK)

# Frequency vs Monetary scatter
seg_colors_map = {'Active': ORANGE,'At Risk': ORANGE2,'Lapsing': YELLOW,'Lost': '#cccccc'}
for seg, grp in rfm.groupby('Segment'):
    axes[1,1].scatter(grp['Frequency'], grp['Monetary'],
                      alpha=0.4, s=18, label=str(seg),
                      color=seg_colors_map.get(str(seg), GRAY))
axes[1,1].set_xlim(0, 60); axes[1,1].set_ylim(0, 20000)
axes[1,1].set_title('Frequency vs Monetary by Segment', fontweight='bold', color=DARK)
axes[1,1].set_xlabel('Frequency (orders)', color=GRAY)
axes[1,1].set_ylabel('Monetary (£)', color=GRAY)
axes[1,1].legend(fontsize=8, markerscale=1.5)

# Top 10 customers
top10 = rfm.nlargest(10, 'Monetary')
axes[1,2].barh(range(10), top10['Monetary'].values, color=PALETTE[:10])
axes[1,2].set_yticks(range(10))
axes[1,2].set_yticklabels([f'Customer {int(c)}' for c in top10['CustomerID']], fontsize=9)
axes[1,2].set_title('Top 10 Customers by Total Spend', fontweight='bold', color=DARK)
axes[1,2].set_xlabel('Total Spend (£)', color=GRAY)

for ax in axes.flat:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('chart_rfm_analysis.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: chart_rfm_analysis.png")
print(f"\nTotal Customers : {rfm['CustomerID'].nunique():,}")
print(f"Active (≤30d)   : {(rfm['Segment']=='Active').sum():,}")
print(f"At Risk (≤90d)  : {(rfm['Segment']=='At Risk').sum():,}")
print(f"Lapsing (≤180d) : {(rfm['Segment']=='Lapsing').sum():,}")
print(f"Lost (>180d)    : {(rfm['Segment']=='Lost').sum():,}")


## kernel_11_products.py

In [ ]:
# ================================================
# KERNEL 11 — EDA Chart: Top Products & Hourly Revenue
# ================================================

ORANGE  = '#E84E1B'
ORANGE2 = '#F7941D'
DARK    = '#1A1A1A'
GRAY    = '#666666'

df_eda['Hour'] = df_eda['InvoiceDate'].dt.hour

top_prod = df_eda.groupby('Description')['TotalPrice'].sum().nlargest(10)
top_qty  = df_eda.groupby('Description')['Quantity'].sum().nlargest(10)
hourly   = df_eda.groupby('Hour')['TotalPrice'].sum()

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.patch.set_facecolor('white')

# ── Top 10 products by revenue ──
colors_pr = [ORANGE if i == 0 else ORANGE2 if i == 1 else '#FBBA13'
             if i == 2 else '#cccccc' for i in range(10)]
axes[0].barh(range(10), top_prod.values[::-1] / 1000, color=colors_pr[::-1])
axes[0].set_yticks(range(10))
axes[0].set_yticklabels([t[:30] for t in top_prod.index[::-1]], fontsize=8)
axes[0].set_title('Top 10 Products by Revenue (£K)',
                  fontsize=12, fontweight='bold', color=DARK)
axes[0].set_xlabel('Revenue (£K)', color=GRAY)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# ── Top 10 products by quantity ──
axes[1].barh(range(10), top_qty.values[::-1], color=[ORANGE2] * 10, alpha=0.85)
axes[1].set_yticks(range(10))
axes[1].set_yticklabels([t[:30] for t in top_qty.index[::-1]], fontsize=8)
axes[1].set_title('Top 10 Products by Quantity Sold',
                  fontsize=12, fontweight='bold', color=DARK)
axes[1].set_xlabel('Total Quantity', color=GRAY)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

# ── Revenue by hour of day ──
axes[2].plot(hourly.index, hourly.values / 1000,
             color=ORANGE, linewidth=2.5, marker='o', markersize=6)
axes[2].fill_between(hourly.index, hourly.values / 1000, alpha=0.15, color=ORANGE)
peak_h = hourly.idxmax()
axes[2].annotate(f'Peak\n{peak_h}:00',
                 xy=(peak_h, hourly[peak_h] / 1000),
                 xytext=(peak_h + 1, hourly[peak_h] / 1000 + 40),
                 arrowprops=dict(arrowstyle='->', color=DARK),
                 fontsize=9, color=DARK, fontweight='bold')
axes[2].set_title('Revenue by Hour of Day (£K)',
                  fontsize=12, fontweight='bold', color=DARK)
axes[2].set_xlabel('Hour', color=GRAY)
axes[2].set_ylabel('Revenue (£K)', color=GRAY)
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('chart_products_hourly.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: chart_products_hourly.png")
print(f"\nTop Product by Revenue : {top_prod.index[0]} — £{top_prod.iloc[0]:,.0f}")
print(f"Top Product by Quantity: {top_qty.index[0]} — {top_qty.iloc[0]:,} units")
print(f"Peak Hour              : {peak_h}:00")


## kernel_12_correlation.py

In [ ]:
# ================================================
# KERNEL 12 — EDA Chart: Correlation & Missing Values
# ================================================

ORANGE  = '#E84E1B'
ORANGE2 = '#F7941D'
DARK    = '#1A1A1A'
GRAY    = '#666666'

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('white')

# ── Correlation heatmap ──
corr = df_eda[['Quantity', 'UnitPrice', 'TotalPrice']].corr()
im   = axes[0].imshow(corr, cmap='Oranges', vmin=0, vmax=1)
axes[0].set_xticks(range(3)); axes[0].set_yticks(range(3))
axes[0].set_xticklabels(corr.columns, rotation=30)
axes[0].set_yticklabels(corr.columns)
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, f'{corr.iloc[i,j]:.2f}',
                     ha='center', va='center', fontweight='bold', fontsize=13)
axes[0].set_title('Correlation Heatmap', fontsize=12, fontweight='bold', color=DARK)
plt.colorbar(im, ax=axes[0])

# ── Missing values bar ──
miss = df.isnull().sum(); miss = miss[miss > 0]
bars = axes[1].bar(miss.index, miss.values, color=[ORANGE, ORANGE2])
axes[1].set_title('Missing Values by Column', fontsize=12, fontweight='bold', color=DARK)
axes[1].set_ylabel('Count', color=GRAY)
for i, v in enumerate(miss.values):
    axes[1].text(i, v + 1000, f'{v:,}', ha='center',
                 fontsize=10, color=DARK, fontweight='bold')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

# ── Cancelled vs Normal orders pie ──
can_n = df['InvoiceNo'].astype(str).str.startswith('C').sum()
nor_n = len(df) - can_n
axes[2].pie([nor_n, can_n],
            labels=['Normal Orders', 'Cancellations'],
            colors=[ORANGE, '#cccccc'],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[2].set_title('Normal vs Cancelled Orders',
                  fontsize=12, fontweight='bold', color=DARK)

plt.tight_layout()
plt.savefig('chart_correlation_missing.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: chart_correlation_missing.png")
print(f"\nCorrelation (Quantity vs TotalPrice) : {corr.loc['Quantity','TotalPrice']:.3f}")
print(f"Correlation (UnitPrice vs TotalPrice): {corr.loc['UnitPrice','TotalPrice']:.3f}")
print(f"Missing CustomerID : {df['CustomerID'].isnull().sum():,}")
print(f"Missing Description: {df['Description'].isnull().sum():,}")
print(f"Cancelled orders   : {can_n:,}  ({can_n/len(df)*100:.1f}%)")


## kernel_13_clean_duplicates.py

In [ ]:
# ================================================
# KERNEL 13 — Cleaning Step 1: Remove Duplicates
# ================================================
# Start fresh for cleaning

df_clean = pd.read_csv('online_retail.csv', encoding='latin1')
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

print("=== Before Cleaning ===")
print(f"Total rows : {len(df_clean):,}")

# Show a sample of duplicate rows
print("\nSample duplicate rows:")
print(df_clean[df_clean.duplicated(keep=False)].head(4))

# Remove duplicates
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after   = len(df_clean)

print(f"\n=== Step 1: Remove Duplicates ===")
print(f"Rows before : {before:,}")
print(f"Rows removed: {before - after:,}")
print(f"Rows after  : {after:,}")


## kernel_14_clean_cancelled.py

In [ ]:
# ================================================
# KERNEL 14 — Cleaning Step 2: Remove Cancelled Orders
# ================================================
# Invoices starting with 'C' are cancellations/refunds
# They are NOT real sales — remove them

print("Sample cancelled orders (Invoice starts with C):")
print(df_clean[df_clean['InvoiceNo'].astype(str).str.startswith('C')].head(4))

before = len(df_clean)
cancelled_df = df_clean[df_clean['InvoiceNo'].astype(str).str.startswith('C')].copy()
df_clean     = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]
after        = len(df_clean)

print(f"\n=== Step 2: Remove Cancelled Orders ===")
print(f"Rows before : {before:,}")
print(f"Rows removed: {before - after:,}")
print(f"Rows after  : {after:,}")
print(f"(Cancelled orders saved separately: {len(cancelled_df):,} rows)")


## kernel_15_clean_missing.py

In [ ]:
# ================================================
# KERNEL 15 — Cleaning Steps 3 & 4: Missing Values
# ================================================

# Step 3: Drop rows with missing CustomerID
# Without CustomerID we can't do customer analysis or churn prediction
print("Missing CustomerID before:", df_clean['CustomerID'].isnull().sum())

before   = len(df_clean)
df_clean = df_clean.dropna(subset=['CustomerID'])
after    = len(df_clean)

print(f"\n=== Step 3: Drop Missing CustomerID ===")
print(f"Rows before : {before:,}")
print(f"Rows removed: {before - after:,}")
print(f"Rows after  : {after:,}")

# Step 4: Fill missing Description with 'Unknown'
# Only a small number remain — no need to drop the whole row
print(f"\nMissing Description before: {df_clean['Description'].isnull().sum()}")
filled   = df_clean['Description'].isnull().sum()
df_clean['Description'] = df_clean['Description'].fillna('Unknown')
print(f"\n=== Step 4: Fill Missing Description ===")
print(f"Filled {filled} values with 'Unknown'")
print(f"Missing Description after: {df_clean['Description'].isnull().sum()}")


## kernel_16_clean_qty_price.py

In [ ]:
# ================================================
# KERNEL 16 — Cleaning Steps 5 & 6: Invalid Qty & Price
# ================================================

# Step 5: Remove negative / zero Quantity
# A real sale must have Quantity > 0
print(f"Rows with Quantity ≤ 0 : {(df_clean['Quantity'] <= 0).sum():,}")
print("Sample negative quantity rows:")
print(df_clean[df_clean['Quantity'] <= 0][['InvoiceNo','Description','Quantity']].head(3))

before   = len(df_clean)
df_clean = df_clean[df_clean['Quantity'] > 0]
after    = len(df_clean)

print(f"\n=== Step 5: Remove Negative / Zero Quantity ===")
print(f"Rows before : {before:,}")
print(f"Rows removed: {before - after:,}")
print(f"Rows after  : {after:,}")

# Step 6: Remove zero / negative UnitPrice
# A product with zero or negative price = bad data
print(f"\nRows with UnitPrice ≤ 0 : {(df_clean['UnitPrice'] <= 0).sum():,}")
print("Sample invalid price rows:")
print(df_clean[df_clean['UnitPrice'] <= 0][['InvoiceNo','Description','UnitPrice']].head(3))

before   = len(df_clean)
df_clean = df_clean[df_clean['UnitPrice'] > 0]
after    = len(df_clean)

print(f"\n=== Step 6: Remove Zero / Negative UnitPrice ===")
print(f"Rows before : {before:,}")
print(f"Rows removed: {before - after:,}")
print(f"Rows after  : {after:,}")


## kernel_17_clean_stockcodes.py

In [ ]:
# ================================================
# KERNEL 17 — Cleaning Step 7: Non-Product StockCodes
# ================================================
# Codes with only letters (POST, DOT, BANK etc.)
# are service charges — not real products

print("Sample non-product StockCodes found:")
non_product_codes = df_clean[
    df_clean['StockCode'].astype(str).str.match(r'^[A-Za-z]+$')
]['StockCode'].unique()
print(non_product_codes[:10])

print(f"\nTotal rows with non-product codes: "
      f"{df_clean['StockCode'].astype(str).str.match(r'^[A-Za-z]+$').sum():,}")

before    = len(df_clean)
is_service = df_clean['StockCode'].astype(str).str.match(r'^[A-Za-z]+$')
df_clean   = df_clean[~is_service]
after      = len(df_clean)

print(f"\n=== Step 7: Remove Non-Product StockCodes ===")
print(f"Rows before : {before:,}")
print(f"Rows removed: {before - after:,}")
print(f"Rows after  : {after:,}")


## kernel_18_clean_dtypes.py

In [ ]:
# ================================================
# KERNEL 18 — Cleaning Step 8: Fix Data Types
# ================================================

print("=== Data Types Before Fixing ===")
print(df_clean.dtypes)

# CustomerID was stored as float (17850.0) — convert to string
df_clean['CustomerID']  = df_clean['CustomerID'].astype(int).astype(str)

# InvoiceNo should be string, not numeric
df_clean['InvoiceNo']   = df_clean['InvoiceNo'].astype(str)

# InvoiceDate already datetime from earlier step
# df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

print("\n=== Data Types After Fixing ===")
print(df_clean.dtypes)

print(f"\nCustomerID sample : {df_clean['CustomerID'].head(3).tolist()}")
print(f"InvoiceNo sample  : {df_clean['InvoiceNo'].head(3).tolist()}")


## kernel_19_clean_outliers.py

In [ ]:
# ================================================
# KERNEL 19 — Cleaning Step 9: Remove Outliers
# ================================================
# Using 1st–99th percentile to cut extreme values
# Quantity = 80,995 and UnitPrice = £38,970 would
# completely distort any forecasting model

print("=== Before Outlier Removal ===")
print(df_clean[['Quantity', 'UnitPrice']].describe())

for col in ['Quantity', 'UnitPrice']:
    low  = df_clean[col].quantile(0.01)
    high = df_clean[col].quantile(0.99)
    print(f"\n{col}:")
    print(f"  1st percentile  : {low:.2f}")
    print(f"  99th percentile : {high:.2f}")
    print(f"  Rows outside    : {((df_clean[col] < low) | (df_clean[col] > high)).sum():,}")

before = len(df_clean)
for col in ['Quantity', 'UnitPrice']:
    low  = df_clean[col].quantile(0.01)
    high = df_clean[col].quantile(0.99)
    df_clean = df_clean[(df_clean[col] >= low) & (df_clean[col] <= high)]
after = len(df_clean)

print(f"\n=== Step 9: Remove Outliers (1st–99th Percentile) ===")
print(f"Rows before : {before:,}")
print(f"Rows removed: {before - after:,}")
print(f"Rows after  : {after:,}")

print("\n=== After Outlier Removal ===")
print(df_clean[['Quantity', 'UnitPrice']].describe())


## kernel_20_feature_eng_save.py

In [ ]:
# ================================================
# KERNEL 20 — Cleaning Step 10: Feature Engineering & Save
# ================================================
# Add useful columns for analysis, dashboard and ML models

df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']
df_clean['Year']       = df_clean['InvoiceDate'].dt.year
df_clean['Month']      = df_clean['InvoiceDate'].dt.month
df_clean['Day']        = df_clean['InvoiceDate'].dt.day
df_clean['Hour']       = df_clean['InvoiceDate'].dt.hour
df_clean['DayOfWeek']  = df_clean['InvoiceDate'].dt.day_name()
df_clean['MonthYear']  = df_clean['InvoiceDate'].dt.to_period('M').astype(str)

print("=== New Columns Added ===")
print("TotalPrice = Quantity × UnitPrice")
print("Year, Month, Day, Hour — extracted from InvoiceDate")
print("DayOfWeek  — Monday, Tuesday...")
print("MonthYear  — 2010-12, 2011-01...")

print(f"\nFinal columns: {df_clean.columns.tolist()}")
print(f"\nSample rows:")
print(df_clean.head(3).to_string())

# Save cleaned data
df_clean.to_csv('online_retail_cleaned.csv', index=False)
print(f"\nCleaned file saved → online_retail_cleaned.csv")

# Final summary
print("\n" + "=" * 50)
print("  CLEANING COMPLETE — FINAL SUMMARY")
print("=" * 50)
print(f"  Original rows  : 541,909")
print(f"  Final rows     : {len(df_clean):,}")
print(f"  Rows removed   : {541909 - len(df_clean):,}")
print(f"  Data retained  : {len(df_clean)/541909*100:.1f}%")
print(f"  Final columns  : {df_clean.shape[1]}")
print(f"  Missing values : {df_clean.isnull().sum().sum()}")
print("=" * 50)


## kernel_21_cleaning_chart.py

In [ ]:
# ================================================
# KERNEL 21 — Cleaning Chart: Before vs After
# ================================================

ORANGE  = '#E84E1B'
ORANGE2 = '#F7941D'
YELLOW  = '#FBBA13'
GREEN   = '#44BBA4'
DARK    = '#1A1A1A'
GRAY    = '#666666'
LO      = '#FDE8E2'
PALETTE = [ORANGE, ORANGE2, YELLOW, '#2E86AB', '#7B4D8E', GREEN]

df_raw = pd.read_csv('online_retail.csv', encoding='latin1')

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.patch.set_facecolor('white')
fig.suptitle('Data Cleaning — Before vs After Comparison',
             fontsize=16, fontweight='bold', color=DARK, y=1.01)

# ── Cleaning funnel ──
step_labels = ['Duplicates\n(5,268)', 'Cancellations\n(9,251)',
               'Missing\nCustomerID\n(134,658)', 'Invalid\nPrice\n(40)',
               'Service\nCodes\n(1,397)', 'Outliers\n(11,332)']
step_vals   = [5268, 9251, 134658, 40, 1397, 11332]

bars_fn = axes[0,0].bar(step_labels, step_vals, color=PALETTE, alpha=0.88, width=0.6)
axes[0,0].set_title('Rows Removed at Each Step', fontweight='bold', color=DARK)
axes[0,0].set_ylabel('Rows Removed', color=GRAY)
axes[0,0].tick_params(axis='x', labelsize=8)
for bar, val in zip(bars_fn, step_vals):
    axes[0,0].text(bar.get_x() + bar.get_width() / 2,
                   bar.get_height() + 300,
                   f'{val:,}', ha='center', fontsize=8,
                   fontweight='bold', color=DARK)
axes[0,0].spines['top'].set_visible(False)
axes[0,0].spines['right'].set_visible(False)

# ── Quantity before vs after ──
axes[0,1].hist(df_raw[df_raw['Quantity'].between(0,100)]['Quantity'],
               bins=40, color='#cccccc', alpha=0.7, label='Before', edgecolor='white')
axes[0,1].hist(df_clean[df_clean['Quantity'] <= 100]['Quantity'],
               bins=40, color=ORANGE, alpha=0.7, label='After', edgecolor='white')
axes[0,1].set_title('Quantity: Before vs After', fontweight='bold', color=DARK)
axes[0,1].set_xlabel('Quantity', color=GRAY)
axes[0,1].set_ylabel('Frequency', color=GRAY)
axes[0,1].legend()
axes[0,1].spines['top'].set_visible(False)
axes[0,1].spines['right'].set_visible(False)

# ── UnitPrice before vs after ──
axes[0,2].hist(df_raw[df_raw['UnitPrice'].between(0,30)]['UnitPrice'],
               bins=40, color='#cccccc', alpha=0.7, label='Before', edgecolor='white')
axes[0,2].hist(df_clean[df_clean['UnitPrice'] <= 30]['UnitPrice'],
               bins=40, color=ORANGE2, alpha=0.7, label='After', edgecolor='white')
axes[0,2].set_title('UnitPrice: Before vs After', fontweight='bold', color=DARK)
axes[0,2].set_xlabel('Unit Price (£)', color=GRAY)
axes[0,2].set_ylabel('Frequency', color=GRAY)
axes[0,2].legend()
axes[0,2].spines['top'].set_visible(False)
axes[0,2].spines['right'].set_visible(False)

# ── Missing values before vs after ──
m_cols = ['CustomerID', 'Description']
b_miss = [df_raw[c].isnull().sum() for c in m_cols]
a_miss = [df_clean[c].isnull().sum() for c in m_cols]
xm = np.arange(2); wm = 0.35
axes[1,0].bar(xm - wm/2, b_miss, wm, label='Before', color='#cccccc')
axes[1,0].bar(xm + wm/2, a_miss, wm, label='After',  color=ORANGE)
axes[1,0].set_xticks(xm); axes[1,0].set_xticklabels(m_cols)
axes[1,0].set_title('Missing Values: Before vs After', fontweight='bold', color=DARK)
axes[1,0].set_ylabel('Count', color=GRAY)
axes[1,0].legend()
axes[1,0].spines['top'].set_visible(False)
axes[1,0].spines['right'].set_visible(False)

# ── TotalPrice (new column) ──
axes[1,1].hist(df_clean[df_clean['TotalPrice'] <= 100]['TotalPrice'],
               bins=50, color=YELLOW, alpha=0.85, edgecolor='white')
axes[1,1].set_title('TotalPrice — New Engineered Column (≤ £100)',
                    fontweight='bold', color=DARK)
axes[1,1].set_xlabel('Total Price (£)', color=GRAY)
axes[1,1].set_ylabel('Frequency', color=GRAY)
axes[1,1].spines['top'].set_visible(False)
axes[1,1].spines['right'].set_visible(False)

# ── Data retention pie ──
axes[1,2].pie([len(df_clean), 541909 - len(df_clean)],
              labels=['Retained', 'Removed'],
              colors=[GREEN, '#cccccc'],
              autopct='%1.1f%%', startangle=90,
              wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1,2].set_title(f'Data Retention — {len(df_clean)/541909*100:.1f}% Kept',
                    fontweight='bold', color=DARK)

plt.tight_layout()
plt.savefig('chart_cleaning_before_after.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: chart_cleaning_before_after.png")
